# Human Axis Classifier — Inference

Classifies 2D brain scan slices as coronal, sagittal, or axial using a ConvNeXt-Tiny backbone.

## Inputs

`SCANS_DIR` is a folder containing the scan images to classify. `MODEL_PATH` points to the trained `.pt` weights file.

In [ ]:
SCANS_DIR = "/path/to/scans"
MODEL_PATH = "/path/to/human_axis_classifier.pt"


In [ ]:
import os
from collections import Counter
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
import tifffile
import warnings
warnings.filterwarnings('ignore')
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import timm

Image.MAX_IMAGE_PIXELS = None
device = torch.device('cpu')

CLASSES = ['coronal', 'sagittal', 'axial']
NUM_CLASSES = len(CLASSES)
MODEL_NAME = 'convnext_tiny'
IMG_SIZE = 320
BATCH_SIZE = 16

print(f"Scans folder: {SCANS_DIR} | exists={os.path.isdir(SCANS_DIR)}")
print(f"Model file: {MODEL_PATH} | exists={os.path.isfile(MODEL_PATH)}")


## Data Loading

In [ ]:
VALID_EXTS = ('.tif', '.tiff', '.png', '.jpg', '.jpeg', '.jfif', '.bmp')

def is_image_file(path):
    return os.path.splitext(path)[1].lower() in VALID_EXTS

def load_image(path):
    ext = os.path.splitext(path)[1].lower()
    if ext in ('.tif', '.tiff'):
        arr = tifffile.imread(path)
        if arr.ndim == 2:
            arr = np.stack([arr] * 3, axis=-1)
        elif arr.ndim == 3 and arr.shape[0] in (1, 3, 4):
            arr = np.transpose(arr, (1, 2, 0))
        if arr.shape[-1] == 4:
            arr = arr[..., :3]
        arr = arr.astype(np.float32)
        arr = (arr - arr.min()) / (arr.max() - arr.min() + 1e-8) * 255
        return Image.fromarray(arr.astype(np.uint8)).convert('RGB')
    return Image.open(path).convert('RGB')


In [ ]:
def gather_images(folder_path):
    if not os.path.isdir(folder_path):
        raise FileNotFoundError(f'Folder not found: {folder_path}')
    files = sorted(os.listdir(folder_path))
    img_files = [f for f in files if not f.startswith('.') and is_image_file(f)]
    records = [{'path': os.path.join(folder_path, f), 'filename': f} for f in img_files]
    return records

records = gather_images(SCANS_DIR)
print(f'Total images found: {len(records)}')
if len(records) == 0:
    raise RuntimeError('No valid images found in the scans folder.')

MEAN = [0.485, 0.456, 0.406]
STD = [0.229, 0.224, 0.225]

infer_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class ScanDataset(Dataset):
    def __init__(self, records, transform):
        self.records = records
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, idx):
        rec = self.records[idx]
        img = load_image(rec['path'])
        img = self.transform(img)
        return img, rec['path'], rec['filename']

ds = ScanDataset(records, infer_tf)
dl = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
print(f'Inference batches: {len(dl)}')


## Model

In [ ]:
class AxisClassifier(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.backbone = timm.create_model(MODEL_NAME, pretrained=False, num_classes=0)
        self.head = nn.Sequential(
            nn.Dropout(0.4),
            nn.Linear(self.backbone.num_features, 128),
            nn.GELU(),
            nn.Dropout(0.2),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)
        return self.head(feats)

def load_model(model_path):
    if not os.path.isfile(model_path):
        raise FileNotFoundError(f'Model file not found: {model_path}')
    model = AxisClassifier(num_classes=NUM_CLASSES).to(device)
    ckpt = torch.load(model_path, map_location=device)
    state_dict = ckpt.get('state_dict', ckpt) if isinstance(ckpt, dict) else ckpt
    model.load_state_dict(state_dict)
    model.eval()
    return model

model = load_model(MODEL_PATH)
print(f'Loaded model from: {MODEL_PATH}')


## Inference

In [ ]:
pred_records = []

with torch.no_grad():
    for imgs, paths, filenames in dl:
        imgs = imgs.to(device)
        logits = model(imgs)
        probs = torch.softmax(logits, dim=1)
        confs, pred_idx = probs.max(dim=1)
        for i in range(len(paths)):
            pidx = int(pred_idx[i].item())
            pred_records.append({
                'path': paths[i],
                'filename': filenames[i],
                'pred_class_idx': pidx,
                'pred_class': CLASSES[pidx],
                'confidence': float(confs[i].item()),
            })

print(f'Inference complete for {len(pred_records)} images.')


## Results

In [ ]:
n_total = len(pred_records)

overall_counts = Counter(r['pred_class'] for r in pred_records)
print('=== Predicted Class Distribution ===')
for c in CLASSES:
    n = overall_counts.get(c, 0)
    pct = (n / n_total) * 100 if n_total else 0.0
    print(f'{c:10s}: {n:5d} ({pct:6.2f}%)')

conf = np.array([r['confidence'] for r in pred_records], dtype=np.float32)
print('\n=== Confidence Stats ===')
print(f'Mean   : {conf.mean():.4f}')
print(f'Median : {np.median(conf):.4f}')
print(f'Std    : {conf.std():.4f}')
print(f'Min    : {conf.min():.4f}')
print(f'Max    : {conf.max():.4f}')

print('\n=== Confidence by Predicted Class ===')
for c in CLASSES:
    vals = [r['confidence'] for r in pred_records if r['pred_class'] == c]
    if len(vals) == 0:
        print(f'{c:10s}: no predictions')
        continue
    vals = np.array(vals, dtype=np.float32)
    print(f'{c:10s}: mean={vals.mean():.4f}  std={vals.std():.4f}  min={vals.min():.4f}  max={vals.max():.4f}')


In [ ]:
import csv

OUTPUT_CSV = "predictions.csv"

with open(OUTPUT_CSV, "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["filename", "path", "pred_class", "confidence"])
    writer.writeheader()
    for r in pred_records:
        writer.writerow({
            "filename": r["filename"],
            "path": r["path"],
            "pred_class": r["pred_class"],
            "confidence": f"{r['confidence']:.4f}",
        })

print(f"Predictions written to {OUTPUT_CSV}")


## Visualizations

In [ ]:
plt.figure(figsize=(7, 4))
counts = [overall_counts.get(c, 0) for c in CLASSES]
sns.barplot(x=CLASSES, y=counts, palette='viridis')
plt.title('Predicted Class Distribution')
plt.xlabel('Predicted Class')
plt.ylabel('Count')
plt.tight_layout()
plt.show()


In [ ]:
k = min(12, len(pred_records))
worst = sorted(pred_records, key=lambda x: x['confidence'])[:k]
if k > 0:
    cols = 4
    rows = (k + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 3.5 * rows))
    axes = np.array(axes).reshape(-1)
    for i, rec in enumerate(worst):
        img = load_image(rec['path'])
        axes[i].imshow(img)
        axes[i].set_title(
            f"{rec['filename']}\nPred: {rec['pred_class']} ({rec['confidence']:.2%})",
            fontsize=8
        )
        axes[i].axis('off')
    for j in range(k, len(axes)):
        axes[j].axis('off')
    plt.suptitle('Lowest-Confidence Predictions', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()


In [ ]:
samples_per_class = 4
for c in CLASSES:
    recs = [r for r in pred_records if r['pred_class'] == c]
    recs = sorted(recs, key=lambda x: x['confidence'], reverse=True)[:samples_per_class]
    if len(recs) == 0:
        continue
    fig, axes = plt.subplots(1, len(recs), figsize=(4 * len(recs), 4))
    axes = np.atleast_1d(axes)
    for ax, rec in zip(axes, recs):
        img = load_image(rec['path'])
        ax.imshow(img)
        ax.set_title(f"{rec['filename']}\n{rec['confidence']:.2%}", fontsize=8)
        ax.axis('off')
    plt.suptitle(f'Highest-Confidence "{c}" Predictions', fontsize=12, fontweight='bold')
    plt.tight_layout()
    plt.show()
